In [1]:
# Download the dataset (Windows compatible)
!curl -O https://storage.googleapis.com/generall-shared-data/startups_demo.json

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0 21.1M    0 44547    0     0  59265      0  0:06:14 --:--:--  0:06:14 59875
 52 21.1M   52 11.2M    0     0  6736k      0  0:00:03  0:00:01  0:00:02 6766k
100 21.1M  100 21.1M    0     0  9749k      0  0:00:02  0:00:02 --:--:-- 9785k


In [2]:
import pandas as pd
df = pd.read_json('startups_demo.json', lines=True)
df.head(5)

,name,images,alt,description,link,city
0,SaferCodes,https://safer.codes/img/brand/logo-icon.png,SaferCodes Logo QR codes generator system form...,QR codes systems for COVID-19.\nSimple tools f...,https://safer.codes,Chicago
1,Human Practice,https://d1qb2nb5cznatu.cloudfront.net/startups...,Human Practice - health care information tech...,Point-of-care word of mouth\nPreferral is a mo...,http://humanpractice.com,Chicago
2,StyleSeek,https://d1qb2nb5cznatu.cloudfront.net/startups...,StyleSeek - e-commerce fashion mass customiza...,Personalized e-commerce for lifestyle products...,http://styleseek.com,Chicago
3,Scout,https://d1qb2nb5cznatu.cloudfront.net/startups...,Scout - security consumer electronics interne...,Hassle-free Home Security\nScout is a self-ins...,http://www.scoutalarm.com,Chicago
4,Invitation codes,https://invitation.codes/img/inv-brand-fb3.png,Invitation App - Share referral codes community,The referral community\nInvitation App is a so...,https://invitation.codes,Chicago


In [11]:
%pip install qdrant-client

from qdrant_client import QdrantClient, models

client = QdrantClient(host="localhost", port=6333)

Note: you may need to restart the kernel to use updated packages.


In [6]:
dense_vector_name = 'dense'
sparse_vector_name = 'sparse'

dense_model_name = 'sentence-transformers/all-MiniLM-L6-v2'
sparse_model_name = 'prithivida/Splade_PP_en_v1'

if not client.collection_exists('startups'):
    client.create_collection(
        collection_name='startups',
        vectors_config = {
            dense_vector_name: models.VectorParams(
                size = client.get_embedding_size(dense_model_name),
                distance = models.Distance.COSINE
            )
        },
        sparse_vectors_config={sparse_vector_name: models.SparseVectorParams()},
    )
    

In [9]:
import json

payload_path = 'startups_demo.json'
documents = []
metadata = []

with open(payload_path) as fd:
    for line in fd:
        obj = json.loads(line)
        description = obj['description']
        dense_documents = models.Document(text=description, model = dense_model_name)
        sparse_documents = models.Document(text=description, model = sparse_model_name)
        documents.append(
            {
                dense_vector_name: dense_documents,
                sparse_vector_name: sparse_documents
            }
        )
        metadata.append(obj)

In [8]:
from tqdm import tqdm
    
client.upload_collection(
        collection_name = 'startups',
        vectors = tqdm(documents),
        payload = metadata,
        parallel = 4,
    )

  0%|          | 0/40474 [00:00<?, ?it/s]

100%|██████████| 40474/40474 [2:00:14<00:00,  5.61it/s]     
